In [ ]:
import numpy as np 
import pandas as pd
import opendatasets as od

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import FeatureUnion
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

In [ ]:
#read the data
train_data = pd.read_csv("train.csv")
test_data = pd.read_csv("test.csv")

X_train, y_train = train_data.loc[:, train_data.columns != 'Survived'], train_data['Survived']

X_train

In [ ]:
#show the NA values
pd.DataFrame(X_train.isnull().sum(), columns=['Missing values'])

In [ ]:
#get the age mean.
X_train.Age.mean()

In [ ]:
#get the age mean with more precise classification
train_data.groupby(['Pclass', 'Sex'])['Age'].agg([len, 'mean'])

In [ ]:
#class for input the age.
class AgeImputer(BaseEstimator, TransformerMixin):
    def __init__(self):
        super().__init__()
        self.age_means_ = {}

    def fit(self, X, y=None):
        self.age_means_ = X.groupby(['Pclass', 'Sex']).Age.mean()

        return self

    def transform(self, X, y=None):
        X_ = X.copy()
        for key, value in self.age_means_.items():
            X_.loc[((np.isnan(X_["Age"])) & (X.Pclass == key[0]) & (X_.Sex == key[1])), 'Age'] = value

        return X_

In [ ]:
#creating the pipeline
numerical_cols = ['Pclass', 'Fare']
categorial_cols = ['Sex', 'Embarked']

model = GradientBoostingClassifier()

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer())
])

categorial_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder())
])

age_transformer = Pipeline(steps=[
    ('imputer', AgeImputer())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('numerical', numerical_transformer, numerical_cols),
        ('categorial', categorial_transformer, categorial_cols)
    ])

pipeline = Pipeline(steps=[
    ('Age Imputer', age_transformer),
    ('preprocess', preprocessor),
    ('model', model)
])

In [ ]:
#find the best parameters in the given parameters.
params = { 
           'model__n_estimators': [300],
           'model__max_depth': [1, 2, 3],
           'model__random_state': [42],
         }

grid_search = GridSearchCV(pipeline, params, cv=10, scoring='accuracy')
grid_search.fit(X_train, y_train)

print('Best accuracy score: {}'.format(grid_search.best_score_))

print('Best params: {}'.format(grid_search.best_params_))

In [ ]:
predictions = grid_search.predict(test_data)

output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('my_submission.csv', index=False)

In [ ]:
print(predictions)

In [ ]:
train_data.info()

In [ ]:
#try a different to treat the data
train_data2 = train_data.drop(['PassengerId','Survived','Ticket','Cabin'], axis=1)
test_data2 = test_data.drop(['PassengerId','Ticket','Cabin'], axis=1)
#concat the train data with the test data
data1 = pd.concat([train_data2, test_data2], axis=0).reset_index(drop=True)
data1.info()
print('-'*40)
data1

In [ ]:
data2 = data1.copy()
#check how many NA data in the dataset
data2.isna().sum()

In [ ]:
data2.info()

In [ ]:
data2['Embarked'] = data2["Embarked"].fillna(data2['Embarked'].mode()[0])

In [ ]:
#cleaning the numeric missing values
def knn_impute(df,na_target):
    df=df.copy()
    
    numeric_df = df.select_dtypes(np.number)
    non_na_columns = numeric_df.loc[:,numeric_df.isna().sum() == 0].columns
    
    y_train = numeric_df.loc[numeric_df[na_target].isna()==False,na_target]
    X_train = numeric_df.loc[numeric_df[na_target].isna()==False,non_na_columns]
    X_test = numeric_df.loc[numeric_df[na_target].isna()==True,non_na_columns]
    
    knn = KNeighborsRegressor()
    knn.fit(X_train,y_train)
    
    y_pred = knn.predict(X_test)
    
    df.loc[df[na_target].isna()==True,na_target] = y_pred
    
    return df

In [ ]:
for column in ['Age','Fare'
]:
    data2 = knn_impute(data2, column)

In [ ]:
data2.isna().sum()

In [ ]:
data2.info()

In [ ]:
data3 = data2.copy()

In [ ]:
data3.info()

In [ ]:
#mapping Sex
data3['Sex'] = data3['Sex'].map({'male':1,'female':0}).astype(int)

#mapping Embarked
data3['Embarked'] = data3['Embarked'].map( {'S': 0, 'C': 1, 'Q': 2} ).astype(int)

In [ ]:
#replace the Name with Title, to change the data type to float64 for more convenient use
data3['Title'] = data3['Name'].str.extract('([A-Za-z]+)\.',expand=False)

data3['Title'].replace(['Mlle','Ms','Lady'], 'Miss',inplace=True)
data3['Title'].replace(['Mme'], 'Mrs',inplace=True)
data3['Title'].replace(['Countess','Capt','Col','Don','Dr',
                        'Major','Rev',"Sir","Jonkheer",'Dona'], 'Rare',inplace=True)
data3.drop(['Name'],inplace=True, axis=1)
Tempdate = data3.copy()
Tempdate['Survived'] = y_train
Tempdate.loc[:train_data.index.max(),:][['Title','Survived']].groupby(['Title'],as_index=False).mean().sort_values(by='Survived',ascending=True)

data3['Title'] = data3['Title'].map({'Master':0,'Miss':1,'Mrs':1,'Mr':2,'Rare':3})

In [ ]:
data4 = data3.copy()

In [ ]:
#standardize data
scaler = StandardScaler()
scaler.fit(data4)
data4 = pd.DataFrame(scaler.transform(data4),index=data4.index,columns=data4.columns)

In [ ]:
#split the data
train_final = data4.loc[:train_data.index.max(), :].copy()
test_final = data4.loc[train_data.index.max() + 1:, :].reset_index(drop=True).copy()

In [ ]:
train_final.info()
print('_'*40)
test_final.info()

In [ ]:
max_features = list(range(1,train_final.shape[1]))

In [ ]:
#find the best parameters in the given parameters.
parameters = {
    'max_features':max_features,
    'n_estimators':[5,10,20,100,250],
    'max_depth':[1,3,5,7,9],
    'learning_rate':[0.01,0.05,0.1]
}


gbc = GridSearchCV(GradientBoostingClassifier(), parameters, cv=10)


gbc.fit(train_final, y_train)

In [ ]:
print(gbc.score(train_final, y_train))
print(gbc.best_params_)

In [ ]:
gbc = GradientBoostingClassifier(learning_rate= 0.05, loss= 'deviance', max_depth=5, n_estimators= 100,max_features=6)
gbc.fit(train_final,y_train)
pred = gbc.predict(train_final)
print(classification_report(y_train,pred))

In [ ]:
final_predictions = gbc.predict(test_final)

output = pd.concat([test_ids,pd.Series(final_predictions,name="Survived")],axis=1)
output.to_csv("./my_submission2.csv",index=False,header=True)